# FashionMNIST Image Classifier with PyTorch

This notebook ties together the helper modules in this folder:

| Step | File | Function(s) |
|---|---|---|
| 0. Hardware check | `hardware_check.py` | `get_device()` |
| 1. Train / validation / test split | `fashion_split.py` | `load_fashion_mnist_splits()` |
| 2. Preprocessing | `fashion_preprocess.py` | `preprocess_splits()` |
| 3. Mini-batch DataLoaders | `fashion_dataloaders.py` | `get_dataloaders()` |
| 4. Build & train the CNN | `fashion_train.py` | `FashionCNN`, `train_model()`, `plot_accuracy()` |
| 5. Test-set evaluation | `fashion_evaluate.py` | `evaluate_model()` |

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch

from hardware_check import get_device
from fashion_split import load_fashion_mnist_splits
from fashion_preprocess import preprocess_splits
from fashion_dataloaders import get_dataloaders
from fashion_train import FashionCNN, train_model, plot_accuracy
from fashion_evaluate import evaluate_model

## 0. Hardware check
Pick the best available device: CUDA (NVIDIA GPU), MPS (Apple Silicon) or CPU.

In [ ]:
device = get_device()

## 1. Split the dataset
The 60,000 official training images are split into **55,000 train** and **5,000 validation** images (seed 42).
The official **10,000 test** images are kept untouched for the final evaluation.

In [ ]:
train_data, valid_data, test_data = load_fashion_mnist_splits()
class_names = test_data.classes
print(f"train: {len(train_data)}, valid: {len(valid_data)}, test: {len(test_data)}")
print("classes:", class_names)

Let's look at some training images:

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, idx in zip(axes.flat, range(16)):
    X, y = train_data[idx]
    ax.imshow(X.squeeze(), cmap="binary")
    ax.set_title(class_names[y], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Preprocessing
- **Standardization**: pixels are scaled with the *training-set* mean and std, which makes optimization faster and more stable.
- **Data augmentation** (train only): random horizontal flips give the model more varied examples and reduce overfitting.

In [ ]:
train_pp, valid_pp, test_pp, (mean, std) = preprocess_splits(train_data, valid_data, test_data)
print(f"training pixel mean: {mean:.4f}, std: {std:.4f}")
X_sample = torch.stack([valid_pp[i][0] for i in range(1000)])
print(f"validation sample after standardization: mean={X_sample.mean():.3f}, std={X_sample.std():.3f}")

## 3. DataLoaders
`get_dataloaders()` runs steps 1 and 2 and wraps each split in a `DataLoader` (batch size 128, shuffling only for training).

In [ ]:
train_loader, valid_loader, test_loader, class_names = get_dataloaders(batch_size=128)
X_batch, y_batch = next(iter(train_loader))
print(f"batches -> train: {len(train_loader)}, valid: {len(valid_loader)}, test: {len(test_loader)}")
print(f"X batch: {tuple(X_batch.shape)} {X_batch.dtype} | y batch: {tuple(y_batch.shape)} {y_batch.dtype}")

## 4. Build and train the model
`FashionCNN` has two convolutional blocks (conv → batchnorm → ReLU ×2, then max-pool) followed by a dropout-regularized dense classifier.

Training uses cross-entropy loss and Adam; the learning rate is halved when validation accuracy plateaus,
and the returned model keeps the weights of the epoch with the **best validation accuracy**.

In [ ]:
print(FashionCNN())

In [ ]:
model, history = train_model(train_loader, valid_loader, n_epochs=15, lr=1e-3, device=device)

### Train vs. validation accuracy per epoch

In [ ]:
plot_accuracy(history)

## 5. Evaluate on the test set

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, class_names)

Save the trained weights so the model can be reloaded later without retraining:

In [ ]:
torch.save(model.state_dict(), "fashion_cnn.pt")

# To reload:
# model = FashionCNN().to(device)
# model.load_state_dict(torch.load("fashion_cnn.pt", map_location=device))